# Landing → Bronze

Ingestão das 8 tabelas — mesma lógica pra todas, só muda o nome do arquivo e o encoding. Baseado no
que o grupo já estava rodando direto no Databricks; só adicionei o encoding certo por arquivo (sem
isso, `clientes` e `Chamados_Hora` saem com acento quebrado — são `ISO-8859-1`, não `UTF-8`) e padronizei
o nome da tabela de destino em minúsculo. Muda só por PR revisado, igual ao notebook base em `analises/`.

Lê direto da pasta do Workspace (`Shared/case-final-rocket-lab/dados`) com o esquema `file:` no caminho
— sem isso, `spark.read` distribuído dá erro de permissão nessa pasta.

- **Fiel à origem**: tudo como `string` (`inferSchema=false`), sem renomear/corrigir nada aqui — isso é
  papel da silver.
- **Idempotente**: `overwrite` — rodar de novo dá o mesmo resultado.
- **Rastreável**: `_source_file`, `_source_file_mod_ts` e `_ingestion_ts` em toda tabela.
- **Sem dado pessoal em output**: nenhuma célula faz `display()`/`show()` de linha de `clientes`
  (CPF/nome/e-mail) — só contagem.

In [ ]:
from pyspark.sql import functions as F

BASE_PATH = "/Workspace/Shared/case-final-rocket-lab/dados"
CATALOGO = "dados_vcredit"
SCHEMA_BRONZE = "default"

# arquivo -> encoding. clientes e Chamados_Hora são ISO-8859-1; os demais UTF-8.
ARQUIVOS = {
    "chamados":            "UTF-8",
    "Chamados_Hora":       "ISO-8859-1",
    "clientes":            "ISO-8859-1",
    "custos":              "UTF-8",
    "pesquisa_satisfacao": "UTF-8",
    "base_atendentes":     "UTF-8",
    "base_motivos":        "UTF-8",
    "canais":              "UTF-8",
}

## Carga + escrita

Uma função só, chamada uma vez por tabela. Tabela de destino sempre em minúsculo
(`<nome>_raw`), mesmo quando o nome do arquivo tem maiúscula (`Chamados_Hora` → `chamados_hora_raw`).

In [ ]:
def ingere(nome_arquivo, encoding):
    df_bronze = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("encoding", encoding)
        .option("quote", '"')
        .option("escape", '"')
        .load(f"file:{BASE_PATH}/{nome_arquivo}.csv")
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file"),
            F.col("_metadata.file_modification_time").alias("_source_file_mod_ts"),
            F.current_timestamp().alias("_ingestion_ts"),
        )
    )

    destino = f"{CATALOGO}.{SCHEMA_BRONZE}.{nome_arquivo.lower()}_raw"
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")               # idempotente: roda de novo, mesmo resultado
        .option("overwriteSchema", "true")
        .saveAsTable(destino)
    )
    print(f"{destino}: {spark.table(destino).count():,} linhas")


for nome_arquivo, encoding in ARQUIVOS.items():
    ingere(nome_arquivo, encoding)

## Conferência

Contagens esperadas (arquivos originais, ver `docs/achados_analise_geral.md`): chamados 134.474 ·
chamados_hora 131.838 · clientes 47.705 · custos 131.069 · pesquisa_satisfacao 81.021 ·
base_atendentes 25 · base_motivos 14 · canais 4.

## Checklist antes de comitar / dar push

- [ ] Contagens acima batem com o esperado.
- [ ] Rodei duas vezes e o resultado foi o mesmo (idempotente).
- [ ] Se precisou mudar algo, foi por PR revisado — não direto na `main`.